# 01 · One spiking neuron

Build and run single spiking neurons with py-spinnaker2: first a neuron with a constant input, then a neuron driven by input spikes through synapses. Everything runs on the Brian2 simulator. Notebook 02 builds a network from these pieces and runs it on the SpiNNaker2 chip.

Work in pairs. **Try** marks exercises for the pair, **Discuss** marks questions we take up together.


The notebook builds on the vendor's tutorials and adds exercises for the workshop. Each section ends with **Go deeper**: where the content comes from and where to read more.

- Vendor tutorials (SpiNNcloud, Apache-2.0): [py-spinnaker2 `tutorials/deep_dive`](https://gitlab.com/spinnaker2/py-spinnaker2/-/tree/v0.8.5/tutorials/deep_dive), referred to as *tutorial 00*, *01*, …
- API documentation: [spinnaker2.gitlab.io/py-spinnaker2](https://spinnaker2.gitlab.io/py-spinnaker2/)
- **Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
# --- libraries ---
import numpy as np  # arrays and numbers
import matplotlib.pyplot as plt  # figures
import ipywidgets as widgets  # the sliders in section 1.1
from spinnaker2 import brian2_sim, snn  # py-spinnaker2: the Brian2 backend and the network objects

# --- backend: where the networks run ---
brian2 = brian2_sim.Brian2Backend()  # the simulator on this computer; notebook 02 runs a network on the chip

# --- figure style ---
plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

## 1. One neuron with a constant input

### 1.1 The physics: the membrane as an RC circuit

The cell membrane separates charge like a capacitor $C$, and ion channels let charge leak through it like a resistor $R$. An input current $I$ charges the membrane, the leak pulls it back to rest. Together this is the leaky integrate-and-fire (LIF) equation in continuous time:

$$\tau_m \frac{dV}{dt} = \underbrace{-(V - V_\text{rest})}_{\text{leak}} + \underbrace{R\,I(t)}_{\text{input}}, \qquad \tau_m = RC$$

As long as $V$ stays below the threshold, the neuron is just this circuit. The cell below switches a constant input on at 25 ms and off at 125 ms. Move the sliders: what does the time constant $\tau_m$ change, and what does the input strength $R\,I$ change? (The threshold comes in the next part.)

<details><summary>How the curve is computed: solving the equation for an input that switches on and off</summary>

Measure $V$ relative to rest, $v = V - V_\text{rest}$, so the equation reads $\tau_m \, dv/dt = -v + R I$. For a constant input $I$, the right-hand side is zero at the steady state $v_\infty = R I$, and writing $u = v - v_\infty$ turns the equation into $\tau_m \, du/dt = -u$. Its solution is an exponential decay, $u(t) = u(t_0)\, e^{-(t - t_0)/\tau_m}$, so in general (Eq. 6 in the workshop note [Spiking Neurons and Networks](../01_knowledge_base/01_primer_snn_basics.md#24-the-lif-equation-term-by-term)):

$$v(t) = v_\infty + \big(v(t_0) - v_\infty\big)\, e^{-(t - t_0)/\tau_m}$$

The cell applies this twice:

- **Input on** ($t_\text{on} \le t < t_\text{off}$): start at rest, $v(t_\text{on}) = 0$, heading for $v_\infty = R I$:
  $$v(t) = R I \left(1 - e^{-(t - t_\text{on})/\tau_m}\right)$$
  After one $\tau_m$, $v$ has covered $1 - e^{-1} \approx 63\,\%$ of the way to $R I$.
- **Input off** ($t \ge t_\text{off}$): now $v_\infty = 0$, and the start value is where the first phase ended, $v(t_\text{off}) = R I \left(1 - e^{-(t_\text{off} - t_\text{on})/\tau_m}\right)$ (called `v_at_switch_off` in the code):
  $$v(t) = v(t_\text{off})\, e^{-(t - t_\text{off})/\tau_m}$$
  The potential leaks back to rest with the same time constant.

</details>

In [ ]:
def plot_rc(tau_m=10.0, RI=1.0):
    """Membrane potential of an RC circuit (LIF below threshold) for a constant input from 25 to 125 ms."""
    t = np.linspace(0, 200, 2001)  # time in ms
    t_on, t_off = 25, 125
    v_at_switch_off = RI * (1 - np.exp(-(t_off - t_on) / tau_m))  # where the charging phase ends
    v = np.where(t < t_on, 0.0,
        np.where(t < t_off, RI * (1 - np.exp(-(t - t_on) / tau_m)),  # charging towards R*I
                 v_at_switch_off * np.exp(-(t - t_off) / tau_m)))  # leaking back to rest

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axvspan(t_on, t_off, color="0.93", label="input on")
    ax.plot(t, v, lw=2, label="membrane potential V")
    ax.axhline(RI, ls="--", c="0.5", label="R·I: where V is heading")
    ax.axvline(t_on + tau_m, ls=":", c="0.5")
    ax.text(t_on + tau_m, 0.02, " one τ: 63 % of the way", va="bottom", color="0.4")
    ax.set(xlim=(0, 200), ylim=(-0.05, 2.1), xlabel="time (ms)", ylabel="V − V_rest (arbitrary units)")
    ax.legend(loc="upper right")
    plt.show()


# --- sliders: move them, the figure updates ---
widgets.interact(
    plot_rc,
    tau_m=widgets.FloatSlider(value=10, min=2, max=50, step=1, description="τ_m (ms)", continuous_update=False),
    RI=widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description="R·I", continuous_update=False),
);

### 1.2 On the chip: in discrete steps

The neuron in 1.1 was a circuit without a threshold, in continuous time. The chip changes three things. Time advances in steps of 1 ms. The leak is a factor per step instead of a time constant: `alpha_decay` = $\alpha = e^{-\Delta t/\tau_m}$, so $\alpha = 0.9$ corresponds to $\tau_m \approx 9.5$ ms. And the neuron gets a threshold: when $v$ crosses it, the neuron fires and $v$ is reset. The constant input of 1.1, $R\,I$, becomes `i_offset`, a number added in every step.

The `lif` neuron of SpiNNaker2 updates its membrane potential once per time step:

$$v[t] = \alpha \, v[t-1] + I_\text{syn}[t] + i_\text{offset}$$

- `alpha_decay` ($\alpha$): how much of the potential is kept per step (0.9 = 10 % leaks away)
- `i_offset`: a constant input added every step
- `threshold`: when $v$ exceeds it, the neuron spikes and $v$ is reset
- `reset`: `"reset_by_subtraction"` subtracts the threshold and keeps the overshoot; `"reset_to_v_reset"` sets $v$ to `v_reset`

The cell below runs one neuron for 50 steps and plots its membrane potential. It uses four pieces of py-spinnaker2, the same on Brian2 and on the chip:

- `snn.Population` creates a group of neurons of one model, here a single `lif` neuron with the parameters above; `record` says what to keep: its spikes and its potential `v`.
- `snn.Network` holds everything that runs together.
- `brian2.run(net, 50)` runs 50 time steps on the backend created in the setup cell. It returns nothing; the results are stored in the population.
- `get_voltages()` and `get_spikes()` read them back, as a dict per neuron index: `[0]` is the first (and here only) neuron. The recorded potential is the value after a possible reset, so the moment of crossing the threshold shows up as a drop, marked with a red triangle.

<details><summary>Derivation of discrete time equation</summary>

**From continuous to discrete time.** 
Complete derivations in the workshop note *Spiking Neurons and Networks*: [LIF in discrete time](../01_knowledge_base/01_primer_snn_basics.md#3-lif-in-discrete-time-as-on-the-chip) (Eq. 9–14). Textbook treatment: [Gerstner et al., *Neuronal Dynamics*, section 1.3](https://neuronaldynamics.epfl.ch/online/Ch1.S3.html).

</details>

In [ ]:
# --- change these ---
params = {"threshold": 10.0, "alpha_decay": 0.9, "i_offset": 1.5, "reset": "reset_by_subtraction"}

# --- build and run ---
neuron = snn.Population(size=1, neuron_model="lif", params=params, name="neuron", record=["spikes", "v"])
net = snn.Network("one_neuron")  # no projection: the neuron only gets i_offset
net.add(neuron)
brian2.run(net, 50)  # 50 time steps; the results are stored in `neuron`

v = neuron.get_voltages()[0]  # neuron 0: v in every step
spikes = neuron.get_spikes()[0]  # neuron 0: the steps in which it fired

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(v, ".-", label="membrane potential v")  # one dot per time step
ax.axhline(params["threshold"], ls="--", c="0.5", label="threshold")
ax.plot(spikes, [params["threshold"]] * len(spikes), "rv", label="spike")
ax.set_xticks(range(51), minor=True)  # one tick per time step: the simulation is discrete
ax.set_xticks(range(0, 51, 5))
ax.set(xlabel="time step (1 step = 1 ms)", ylabel="v (threshold = 10)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)  # above the plot, clear of the curve
plt.show()
print(f"{len(spikes)} spikes in 50 steps")

**Try (before scrolling down):** the next cell shows the answers, so please don't peek before you have your own. Work with the cell above.

1. Set `i_offset` to 0.5. Why does the neuron never spike? *Hint:* without spikes, $v$ approaches $i_\text{offset} / (1 - \alpha)$.
2. Work out on paper the smallest `i_offset` that makes the neuron spike with `alpha_decay = 0.9`. Then check it in the cell above.
3. *(if time)* Predict how the number of spikes in 1,000 steps changes as `i_offset` grows from 0 to 4. Sketch the curve: where does it start, does it jump, is it straight or bent?

*Your answers:*

1. …
2. …
3. …

**Spike count vs. input** *(if time; otherwise the instructor shows the result after the hands-on)*. The cell below runs 81 neurons at once, each with its own `i_offset` from 0 to 4 (a list with one value per neuron) in steps of 0.05, all other parameters as above, and counts their spikes in 1,000 steps. The red dot is the neuron of the figure above. The dashed line marks the critical input $(1 - \alpha) \cdot$ `threshold` = 1.0: per step, the input only has to replace what leaks away, and it has to exceed this value, not just reach it. Compare with your sketch.

In [ ]:
# --- change these ---
i_offsets = np.round(np.arange(0, 4.01, 0.05), 2)  # one value per neuron
steps = 1000

# --- build and run ---
pop = snn.Population(size=len(i_offsets), neuron_model="lif", params={**params, "i_offset": list(i_offsets)}, name="neurons", record=["spikes"])  # one neuron per i_offset
net = snn.Network("input_output")
net.add(pop)
brian2.run(net, steps)

counts = [len(pop.get_spikes().get(i, [])) for i in range(len(i_offsets))]
critical = (1 - params["alpha_decay"]) * params["threshold"]  # below this input, v never reaches the threshold

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(i_offsets, counts, "o", color="black", ms=4)
nearest = np.argmin(np.abs(i_offsets - params["i_offset"]))  # the neuron closest to the figure above
ax.plot(i_offsets[nearest], counts[nearest], "o", color="red", ms=7)
ax.axvline(critical, ls="--", c="0.5")
ax.text(critical, max(counts), " critical input", va="top", color="0.4")
ax.set(xlabel="i_offset (added per step; threshold = 10)", ylabel=f"spikes in {steps} steps")
plt.show()

<details><summary>Why the curve has small steps</summary>

The neuron can only fire at the tick of a clock, once per time step. So it cannot fire every 3.2 steps; it can only fire in patterns of whole steps, such as "every 3 steps" or "3 times in 10 steps".

Raise the input a little, and the neuron would like to fire a little earlier. But earlier means a whole step earlier, and the small extra input isn't enough for that. So it keeps firing on the same steps as before, and the count stays the same: a flat step in the curve.

The leak makes this stick. A bit of extra charge doesn't pile up, because part of it leaks away every step. Only when the extra input is big enough to reach the threshold a whole step earlier does the neuron switch to a faster pattern, and the count jumps.

Without a leak, the extra charge would pile up until it produced an extra spike, and the count would rise evenly, without steps.

</details>

**Try (if time):** set `alpha_decay` to 0.8 in the first cell of this section and run both cells again. Where does the critical input move, and why?

**Discuss:** the spike count as a function of the input is the neuron's input–output curve. Describe its shape in one sentence: where does it start, and how does it grow? Have you seen a curve like this before?

*Your answers:*

- Try (`alpha_decay` 0.8): …
- Discuss: …

**Go deeper:** tutorial 01, section 3.2 traces the same curve (the f–I curve) by giving every neuron its own `i_offset`; section 3 explains shared, per-neuron and global parameters. All parameters of `lif`: [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html#leaky-integratefire-neuron-lif). Workshop notes: [from Hodgkin-Huxley to LIF](../01_knowledge_base/01_primer_snn_basics.md#2-from-hodgkin-huxley-to-lif) and [LIF in discrete time, as on the chip](../01_knowledge_base/01_primer_snn_basics.md#3-lif-in-discrete-time-as-on-the-chip).

## 2. Input spikes, weights and delays

Neurons usually get their input as spikes from other neurons. Two new building blocks:

- A `spike_list` population sends spikes at given time steps, written as `{neuron index: [time steps]}`. This is also how data enters a network.
- A `Projection` connects two populations. Each connection is one line `[pre, post, weight, delay]`: `pre` is the sending neuron, `post` the receiving one, the **weight** says how strongly one spike pushes the potential (positive up, negative down), the **delay** how many extra steps the spike travels. For `lif`, weights are whole numbers from −15 to 15 and delays from 0 to 7.

Below, three input neurons connect to one post neuron. Input 0 sends three spikes close together, inputs 1 and 2 one spike each, two steps apart. Every weight starts at 6 and every delay at 0. The post neuron leaks a bit faster than in section 1 (`alpha_decay` 0.8 instead of 0.9), so that the timing of its inputs matters more. Change only the lines under `# --- change these ---`.

In [ ]:
# --- change these ---
input_times = {0: [5, 7, 9], 1: [30], 2: [32]}  # input neuron -> time steps of its spikes
connections = [
    [0, 0, 6, 0],  # [pre, post, weight, delay]: input 0 -> post neuron
    [1, 0, 6, 0],  # input 1 -> post neuron
    [2, 0, 6, 0],  # input 2 -> post neuron
]
post_params = {"threshold": 10.0, "alpha_decay": 0.8, "reset": "reset_by_subtraction"}
steps = 50

# --- build and run ---
n_inputs = len(input_times)
inp = snn.Population(size=n_inputs, neuron_model="spike_list", params=input_times, name="input")  # sends the spikes of input_times
post = snn.Population(size=1, neuron_model="lif", params=post_params, name="post", record=["spikes", "v"])
net = snn.Network("synapses")
net.add(inp, post, snn.Projection(pre=inp, post=post, connections=connections))
brian2.run(net, steps)
post_spikes = post.get_spikes()[0]  # the steps in which the post neuron fired
print("post neuron spikes at:", post_spikes)

# --- plot ---
label = {pre: f"input {pre} (w {w}, d {d})" for pre, _, w, d in connections}  # weight and delay per input, for the labels
fig, (ax_in, ax_v, ax_out) = plt.subplots(3, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [1, 3, 1]})

ax_in.eventplot([input_times[i] for i in range(n_inputs)], colors="grey", lineoffsets=range(n_inputs), linelengths=0.7)
ax_in.set(yticks=range(n_inputs), yticklabels=[label.get(i, f"input {i} (not connected)") for i in range(n_inputs)],
          ylim=(-0.6, n_inputs - 0.4), title="input spikes, as sent")

ax_v.plot(post.get_voltages()[0], ".-")
ax_v.axhline(post_params["threshold"], ls="--", c="0.5")
ax_v.text(0, post_params["threshold"] + 0.2, "threshold", va="bottom", color="0.4")
ax_v.set(ylabel="v (threshold = 10)", title="membrane potential of the post neuron")

ax_out.eventplot([post_spikes], colors="black", lineoffsets=0, linelengths=0.7)
ax_out.set(yticks=[0], yticklabels=["post"], ylim=(-0.6, 0.6), title="output spikes")

ax_out.set_xticks(range(steps + 1), minor=True)  # one tick per time step
ax_out.set_xticks(range(0, steps + 1, 5))
ax_out.set(xlim=(-0.5, steps + 0.5), xlabel="time step (1 step = 1 ms)")
plt.tight_layout()
plt.show()

**Try:** predict the result first, then run. Write your answer in the cell below before you move on; we compare the results together afterwards. If time is short, do 1, 3, 4 and 6.

1. One input spike is not enough to reach the threshold. Which input makes the post neuron fire: the three spikes of input 0, or the two spikes of inputs 1 and 2? Why?
2. *(if time)* Change `alpha_decay` to 0.95 and to 0.5. What happens, and why?
3. Change the weight of input 1 to 11. What happens? Then set it back to 6 and give input 2 a weight of −6. What does a negative weight do?
4. Give input 1 a delay of 2. In which step does its spike reach the post neuron, and what happens then?
5. *(if time)* Set the delay of input 1 to 9. In which step does its spike arrive? Compare with what you found in 4.
6. Add a fourth input neuron that fires once at step 20, with its own connection to the post neuron. Which two lines do you have to add or change? Then look at steps 31 to 33: does the new input change what inputs 1 and 2 do, although it fired more than ten steps earlier?

*Your answers:*

1. …
2. …
3. …
4. …
5. …
6. …

**Discuss:**

- In a deep network, a neuron's output depends only on its current inputs, not on when they arrived. Here it does. What role does the leak play in that?
- Spikes sent at different times can arrive together if their delays differ. What could a neuron that only fires for coinciding inputs detect with that? Notebook 02 builds a network on this idea.

*Your answers:*

- …
- …

<details><summary>Background, after the exercises: when a spike arrives, and what happens to values outside the range</summary>

py-spinnaker2 computes the network step by step. Within a step, every neuron is updated and the spikes it sends are collected; they are delivered at the start of the next step. So a spike sent in step $t$ reaches its target in step $t + 1 + \text{delay}$: even with delay 0 it takes one step. On the chip, the next step is when the receiving core processes the spikes that arrived; the Brian2 backend reproduces this, so both give the same spike times. Source: tutorial 02, section 2 (delay line).

Weights and delays outside their range are not rejected: fractions are cut off and values beyond the limits are clipped (a weight of 20 becomes 15, a delay of 9 becomes 7), without a warning, on Brian2 as on the chip. Where these limits come from follows in the hardware block.

</details>

**Go deeper:** tutorial 00, section 5 builds the same kind of network with one excitatory and one inhibitory input and explains both reset rules. Tutorial 02, section 2 builds a delay line and shows the `delay + 1` rule. Workshop notes: [synapses](../01_knowledge_base/01_primer_snn_basics.md#4-synapses), with a figure of the delay, and [where Brian2 and the chip differ](../01_knowledge_base/03_brian2_fallback.md#3-differences-from-the-chip).

---

**End of notebook 01.** Back to the slides: we collect what you found, then continue with the SpiNNaker2 hardware. Notebook 02 follows after that block.